# Functions Are Objects

In [ ]:
states = ["  Alabama", "Georgia!", "Georgia", "georgia", "FlOrIda", "south  carolina##", "West virginia?"]

In [ ]:
import re
def clean_strings(strings): 
    result = []
    for value in strings: 
        value = value.strip()
        value = re.sub("[!#?]", "", value)
        value = value.title()
        result.append(value)
    return result

In [ ]:
print(clean_strings(states))

## Lambda Functions

A **lambda** is an anonymous, single-expression function — a shorthand way to define a small function inline, without giving it a `def` and a name.

```python
lambda arguments: expression
```

It behaves like a regular function (it takes arguments and returns a value) but with two restrictions: it can only contain **one expression** (no statements, no multiple lines, no `for`/`if` blocks — only an expression, though a ternary `x if cond else y` is allowed), and that expression's result is **automatically returned** — there's no `return` keyword.

In practice, lambdas are almost always used as a throwaway function passed *into* another function — `sorted(..., key=...)`, `map(...)`, `filter(...)`, `df.apply(...)` — for logic simple enough that writing a full named `def` would just add clutter. If the logic is reused elsewhere or spans more than one expression, a regular named function is the better (and more readable/debuggable) choice.

### Example 1 — Data Analyst: sorting records by a key with `sorted()`

You have a list of `(region, revenue)` tuples pulled from a query and want them ranked by revenue, highest first. `sorted()` needs a `key` function that, given one record, returns the value to sort on — that's exactly a one-expression job, so a lambda is a natural fit instead of writing a whole named function just to say "grab the revenue."

In [ ]:
regional_revenue = [("North", 48200), ("South", 61500), ("East", 39750), ("West", 72100)]

# key=lambda record: record[1] tells sorted() to compare tuples by their
# second element (revenue) instead of the default (alphabetical on region).
ranked = sorted(regional_revenue, key=lambda record: record[1], reverse=True)
print(ranked)

**What's happening:** for every tuple in `regional_revenue`, `sorted()` calls the lambda with that tuple as `record` and uses the returned value (`record[1]`, the revenue number) as the sort key — the tuples themselves are still what get reordered, only the *comparison* is based on revenue. `reverse=True` flips it to descending, so `West` (72,100) comes first.

### Example 2 — Data Analyst: transforming a pandas column with `.apply()`

You've pulled a `price` column and need a derived `price_tier` column for a report. The transformation is a single expression per row (a ternary), so rather than defining `def price_tier(price): ...` elsewhere in the notebook, a lambda passed straight into `.apply()` keeps the logic next to where it's used.

In [ ]:
import pandas as pd

orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "price": [12.50, 89.99, 45.00, 150.00]})

# .apply() calls the lambda once per value in the "price" column.
orders["price_tier"] = orders["price"].apply(lambda price: "high" if price >= 100 else ("mid" if price >= 40 else "low"))
print(orders)

**What's happening:** `orders["price"].apply(...)` runs the lambda once for each value in the `price` column, passing that single value in as `price`. The nested ternary (`"high" if ... else ("mid" if ... else "low")`) is still one expression, which is why it's legal inside a lambda. The returned strings become the new `price_tier` column, aligned back to the original rows.

### Example 3 — Data Engineer: filtering malformed records in an ingestion step with `filter()`

You're writing a small ETL step that ingests raw JSON-like records before loading them into a warehouse table, and want to drop any record missing a required `user_id`. `filter()` takes a function that returns `True`/`False` for each item and keeps only the ones that pass — a one-line lambda is enough to express "does this record have a user_id."

In [ ]:
raw_records = [
    {"user_id": 101, "event": "click"},
    {"user_id": None, "event": "click"},
    {"event": "view"},                     # missing "user_id" key entirely
    {"user_id": 104, "event": "purchase"},
]

# filter() calls the lambda on every record and keeps only the ones where it returns True.
clean_records = list(filter(lambda record: record.get("user_id") is not None, raw_records))
print(clean_records)

**What's happening:** `record.get("user_id")` safely returns `None` whether the key is missing entirely or explicitly set to `None`, so the lambda's `is not None` check catches both bad cases in one expression. `filter()` runs that check against each dict in `raw_records` and `list(...)` collects only the records where it evaluated to `True` — the record with a missing key and the one with `user_id: None` are both dropped.

### Example 4 — Data Engineer: normalizing keys with `map()` before loading

Downstream systems expect column/field names in `snake_case`, but the source API returns `camelCase` keys. Before loading, you want to standardize each record's keys. `map()` applies a function to every item in an iterable and returns the transformed results — here the "item" is a whole record, and the lambda rebuilds it with normalized keys.

In [ ]:
api_records = [
    {"userId": 1, "eventType": "click"},
    {"userId": 2, "eventType": "purchase"},
]

# map() calls the lambda on each dict in api_records; the lambda's body is a dict
# comprehension (still a single expression) that rebuilds the dict with snake_case keys.
normalized = list(map(lambda record: {re.sub(r"(?<!^)(?=[A-Z])", "_", k).lower(): v for k, v in record.items()}, api_records))
print(normalized)

**What's happening:** for each dict in `api_records`, the lambda receives it as `record` and builds a brand-new dict via a comprehension: `re.sub(r"(?<!^)(?=[A-Z])", "_", k)` inserts an underscore before every capital letter that isn't the first character (e.g., `"userId"` → `"user_Id"`), and `.lower()` finishes the job (`"user_id"`). `map()` does this for every record and `list(...)` materializes the lazy map object into an actual list ready to load.